# GWM-E Link Prediction Training (Cross-Attention - Google Colab)

Train the cross-attention architecture on Google Colab with Google Drive storage.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
import os

# Mount Google Drive
drive.mount('/content/drive')

# Create project directory in Google Drive
PROJECT_ROOT = '/content/drive/MyDrive/NLP-research/code/GWM/link-prediction/'
os.makedirs(PROJECT_ROOT, exist_ok=True)
os.makedirs(f'{PROJECT_ROOT}/data/cora', exist_ok=True)
os.makedirs(f'{PROJECT_ROOT}/trained/cross-attn/cora', exist_ok=True)

print(f"✓ Google Drive mounted")
print(f"✓ Project directory: {PROJECT_ROOT}")

## 2. Install Dependencies

In [ ]:
# Install required packages
!pip uninstall -y protobuf
!pip install -q protobuf==3.20.3
!pip install -q transformers>=4.35.0 accelerate sentencepiece huggingface-hub tqdm matplotlib

print("✓ All dependencies installed")

## 3. Check GPU and Environment

In [ ]:
import torch
import sys

print("="*70)
print(" "*20 + "COLAB ENVIRONMENT")
print("="*70)
print(f"Python version: {sys.version.split()[0]}")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    print(f"CUDA version: {torch.version.cuda}")
else:
    print("⚠️  WARNING: No GPU detected! Training will be very slow.")
    print("   Go to Runtime > Change runtime type > Hardware accelerator > GPU")

print("="*70)

## 4. Configuration

**IMPORTANT:** Upload your training data to Google Drive first!

Expected structure:
```
MyDrive/GWM-E/data/cora/
├── cora_train_link_data.jsonl
├── train_edge_embeddings.pt
├── cora_val_link_data.jsonl
├── val_edge_embeddings.pt
├── cora_test_link_data.jsonl
└── test_edge_embeddings.pt
```

In [ ]:
# ==============================================================================
# DATA PATHS CONFIGURATION (Google Drive)
# ==============================================================================
DATA_DIR = f'{PROJECT_ROOT}/data/cora'

TRAIN_JSONL = f'{DATA_DIR}/cora_train_link_data.jsonl'
TRAIN_EMBEDDING = f'{DATA_DIR}/train_edge_embeddings.pt'
VAL_JSONL = f'{DATA_DIR}/cora_val_link_data.jsonl'
VAL_EMBEDDING = f'{DATA_DIR}/val_edge_embeddings.pt'
TEST_JSONL = f'{DATA_DIR}/cora_test_link_data.jsonl'
TEST_EMBEDDING = f'{DATA_DIR}/test_edge_embeddings.pt'

# Output directory on Google Drive (persistent storage)
OUTPUT_DIR = f'{PROJECT_ROOT}/trained/cross-attn/cora'

# ==============================================================================
# TRAINING HYPERPARAMETERS (CROSS-ATTENTION ARCHITECTURE)
# ==============================================================================
# Model
LLAMA_MODEL = 'meta-llama/Llama-3.2-3B-Instruct'
GRAPH_EMBEDDING_DIM = 768
PROJECTOR_HIDDEN_DIM = 3072  # Cross-attn uses 3072, baseline uses 2048
NUM_HOPS = 4

# Training
BATCH_SIZE = 2
GRADIENT_ACCUMULATION_STEPS = 16
LEARNING_RATE = 3e-5
WEIGHT_DECAY = 0.1
NUM_EPOCHS = 20  # Increased for full training
WARMUP_STEPS = 50
EARLY_STOPPING_PATIENCE = 5
DROPOUT = 0.1
MAX_GRAD_NORM = 1.0
USE_FP16 = True

# Data loading
NUM_WORKERS = 2

# ==============================================================================
# RESUME TRAINING (Set to True to continue from checkpoint)
# ==============================================================================
RESUME_TRAINING = False  # Set to True to resume from checkpoint
CHECKPOINT_DIR = None    # Leave None to auto-detect from OUTPUT_DIR

# Verify data files exist
import os
data_files = [
    TRAIN_JSONL, TRAIN_EMBEDDING,
    VAL_JSONL, VAL_EMBEDDING,
    TEST_JSONL, TEST_EMBEDDING
]

missing_files = [f for f in data_files if not os.path.exists(f)]

if missing_files:
    print("❌ MISSING DATA FILES:")
    for f in missing_files:
        print(f"   {f}")
    print(f"\n📤 Please upload your data files to: {DATA_DIR}")
    print("   Use the Files panel on the left to upload to Google Drive")
else:
    print("="*70)
    print(" "*10 + "GWM-E CROSS-ATTENTION LINK PREDICTION CONFIGURATION")
    print("="*70)
    print(f"\n📊 Architecture: CROSS-ATTENTION")
    print(f"   Split [4, 768] → Bi-attention → Concat → [4096] → MLP projector → LLaMA")
    print(f"\n📁 Data Directory: {DATA_DIR}")
    print(f"💾 Output Directory: {OUTPUT_DIR}")
    print(f"   (Saved to Google Drive - persistent across sessions)")
    print(f"\n🎯 Training:")
    print(f"   Epochs: {NUM_EPOCHS}")
    print(f"   Batch size: {BATCH_SIZE}")
    print(f"   Gradient accumulation: {GRADIENT_ACCUMULATION_STEPS}")
    print(f"   Effective batch: {BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS}")
    print(f"   Learning rate: {LEARNING_RATE}")
    print(f"   Projector hidden dim: {PROJECTOR_HIDDEN_DIM} (cross-attention)")
    print(f"   FP16: {USE_FP16}")
    print(f"\n✓ All data files found!")
    print("="*70)

## 5. Copy Training Files from GitHub

In [ ]:
required_files = ['model.py', 'dataset.py', 'inference.py', 'train.py', 'utils.py']

print("="*70)
print("Cloning GitHub repository...")
print("="*70)

# Clone your GitHub repo
GITHUB_REPO = "https://github.com/HiIamPhuc/GWM.git"
BRANCH = "main"

!git clone {GITHUB_REPO} /content/gwm
%cd /content/gwm
!git checkout {BRANCH}
!git pull
%cd /content

# Copy files from cross-attn folder to working directory
repo_path = "/content/gwm/gwm/link-prediction/cross-attn"

print(f"\nCopying cross-attention files from {repo_path}...")
for file in required_files:
    !cp {repo_path}/{file} /content/
    print(f"✓ Copied {file}")

# Verify files exist
import os
missing_files = [f for f in required_files if not os.path.exists(f)]

if missing_files:
    print(f"\n❌ Missing files: {missing_files}")
    raise FileNotFoundError(f"Required files not found: {missing_files}")
else:
    print(f"\n✓ All required files ready: {required_files}")
    print("✓ Using CROSS-ATTENTION architecture")

## 6. Authenticate with Hugging Face

**Setup HF Token:**
1. Get your token from: https://huggingface.co/settings/tokens
2. In Colab, go to: 🔑 Secrets (left sidebar)
3. Add new secret: Name = `HF_TOKEN`, Value = your token
4. Enable notebook access (toggle switch)

In [ ]:
from google.colab import userdata

try:
    HF_TOKEN = userdata.get('HF_TOKEN')
    !huggingface-cli login --token {HF_TOKEN}
    print("✓ Logged in to Hugging Face")
except Exception as e:
    print(f"❌ Error: {e}")
    print("\n⚠️  Please add HF_TOKEN to Colab Secrets:")
    print("   1. Click 🔑 Secrets in left sidebar")
    print("   2. Add secret: HF_TOKEN = your_token")
    print("   3. Enable notebook access")
    print("   4. Get token from: https://huggingface.co/settings/tokens")

## 7. Train Model (Command Line)

Run training using command-line interface. Models are saved to Google Drive for persistence.

In [ ]:
# Build command with all parameters
cmd = f"""python train.py \\
    --train_jsonl {TRAIN_JSONL} \\
    --train_embedding {TRAIN_EMBEDDING} \\
    --val_jsonl {VAL_JSONL} \\
    --val_embedding {VAL_EMBEDDING} \\
    --test_jsonl {TEST_JSONL} \\
    --test_embedding {TEST_EMBEDDING} \\
    --output_dir {OUTPUT_DIR} \\
    --llama_model {LLAMA_MODEL} \\
    --graph_embedding_dim {GRAPH_EMBEDDING_DIM} \\
    --projector_hidden_dim {PROJECTOR_HIDDEN_DIM} \\
    --num_hops {NUM_HOPS} \\
    --dropout {DROPOUT} \\
    --batch_size {BATCH_SIZE} \\
    --gradient_accumulation_steps {GRADIENT_ACCUMULATION_STEPS} \\
    --lr {LEARNING_RATE} \\
    --weight_decay {WEIGHT_DECAY} \\
    --epochs {NUM_EPOCHS} \\
    --warmup_steps {WARMUP_STEPS} \\
    --max_grad_norm {MAX_GRAD_NORM} \\
    --early_stopping_patience {EARLY_STOPPING_PATIENCE} \\
    --num_workers {NUM_WORKERS}"""

# Add optional flags
if USE_FP16:
    cmd += ''' \\
    --use_fp16'''
if RESUME_TRAINING:
    cmd += ''' \\
    --resume'''
    if CHECKPOINT_DIR:
        cmd += f''' \\
    --checkpoint_dir {CHECKPOINT_DIR}'''

print("Running cross-attention training command:")
print("="*70)
print(cmd)
print("="*70 + "\n")

# Execute training
!{cmd}

print("\n✓ Training completed!")
print(f"📁 Models saved to: {OUTPUT_DIR}")
print("   (Accessible from Google Drive in future sessions)")

## 8. Visualize Results

In [ ]:
import json
import matplotlib.pyplot as plt
from pathlib import Path

# Load training history
history_path = Path(OUTPUT_DIR) / "training_history.json"
results_path = Path(OUTPUT_DIR) / "final_results.json"

if history_path.exists():
    with open(history_path, 'r') as f:
        training_history = json.load(f)
    
    with open(results_path, 'r') as f:
        final_results = json.load(f)
    
    # Extract metrics
    epochs = [h['epoch'] for h in training_history]
    train_losses = [h['train_loss'] for h in training_history]
    val_accuracies = [h['val_accuracy'] for h in training_history]
    test_accuracy = final_results['test_accuracy']
    
    # Create plots
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
    
    # Loss plot
    ax1.plot(epochs, train_losses, 'b-o', label='Train Loss')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Loss')
    ax1.set_title('Training Loss (Cross-Attention - Colab)')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Accuracy plot
    ax2.plot(epochs, [acc * 100 for acc in val_accuracies], 'g-o', label='Validation Accuracy')
    ax2.axhline(y=test_accuracy * 100, color='r', linestyle='--', 
                label=f'Test Accuracy: {test_accuracy*100:.2f}%')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Accuracy (%)')
    ax2.set_title('Validation Accuracy (Cross-Attention - Colab)')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/training_curves.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print("\n" + "="*70)
    print(" "*20 + "FINAL RESULTS (CROSS-ATTENTION)")
    print("="*70)
    print(f"Architecture: {final_results.get('architecture', 'cross-attention')}")
    print(f"Best Validation Accuracy: {final_results['best_val_accuracy']:.4f} ({final_results['best_val_accuracy']*100:.2f}%) at epoch {final_results['best_epoch']}")
    print(f"Final Test Accuracy:      {test_accuracy:.4f} ({test_accuracy*100:.2f}%)")
    print(f"Total epochs trained:     {final_results['total_epochs']}")
    print("="*70)
    print(f"\n📊 Training curves saved to: {OUTPUT_DIR}/training_curves.png")
else:
    print(f"❌ Training history not found at: {history_path}")
    print("   Make sure training has completed successfully.")

## 9. Output Files Summary

All files are saved to Google Drive and will persist across sessions.

In [ ]:
import os
from pathlib import Path

output_dir = Path(OUTPUT_DIR)

if output_dir.exists():
    print("="*70)
    print(" "*15 + "OUTPUT FILES (SAVED TO GOOGLE DRIVE)")
    print("="*70)
    print(f"\n📁 Output directory: {output_dir}\n")
    
    print("💾 Saved Files:")
    total_size = 0
    for file in sorted(output_dir.glob("*")):
        if file.is_file():
            size = os.path.getsize(file) / (1024**2)
            total_size += size
            print(f"  • {file.name:40s} ({size:8.1f} MB)")
    
    print(f"\n📊 Total size: {total_size:.1f} MB")
    print(f"\n✓ All files saved to Google Drive")
    print(f"✓ Accessible in future sessions")
    print(f"✓ To resume training, set RESUME_TRAINING = True")
    print("\n" + "="*70)
else:
    print(f"❌ Output directory not found: {output_dir}")

## 💡 Tips for Using Colab

**Resume Training:**
- Set `RESUME_TRAINING = True` in Configuration cell
- Your checkpoint is saved in Google Drive
- You can stop and resume training anytime

**Monitor GPU Usage:**
- Click "Runtime" > "Manage sessions" to see RAM/GPU usage
- Colab Pro gives you longer sessions and better GPUs

**Save Checkpoints Frequently:**
- Colab sessions can disconnect unexpectedly
- Your checkpoints are safe in Google Drive

**Download Results:**
- All files are in Google Drive: `MyDrive/GWM-E/trained_models/`
- You can access them from any device